## Import Packages

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
print(tf.__version__)

In [ ]:
# Access to Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Prepare the selected features
* The 30 features that are most difficult to distinguish (with the highest p-value) will be trained for the hyperparameter grid search.

In [ ]:
# Load feature data (270) and P-values
FeatureData = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SavedFiles/FeatureData.csv', header=None)
P_value_Rank = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/SavedFiles/P_value_Rank.csv' , header=None)

# Select the 30 features that are most difficult to distinguish (with the highest p-value)
StartRank, Number = 240, 30
FeatureSelected = np.zeros((Number,FeatureData.shape[1]))
s = 0
for i in range(StartRank, StartRank+Number):
    index                = int(P_value_Rank.iloc[i-1,0])
    FeatureSelected[s,:] = FeatureData.iloc[index,:].values
    s += 1

# Standardize the selected features
FeatureSelected_std = StandardScaler().fit_transform(FeatureSelected.T)
FeatureSelected_std.shape

## Split Dataset into Training and Test Sets

In [ ]:
# Separate the dataset into normal and abnormal sets
NormalSet   = FeatureSelected_std[:180 , :]
AbnormalSet = FeatureSelected_std[180: , :]

NormalSet.shape, AbnormalSet.shape

In [ ]:
from sklearn.model_selection    import train_test_split

# Define the test data ratio
TestData_Ratio = 0.2

# Split the normal and abnormal sets into training and test sets
TrainData_Nor, TestData_Nor = train_test_split(NormalSet  , test_size=TestData_Ratio, random_state=777)
TrainData_Abn, TestData_Abn = train_test_split(AbnormalSet, test_size=TestData_Ratio, random_state=777)

print(TrainData_Nor.shape, TestData_Nor.shape)
print(TrainData_Abn.shape, TestData_Abn.shape)

## Label the data (One-hot Encoding) using np.zeros and np.ones
- `[1,0]` refers to 'Normal' and `[0,1]` refers to 'Abnormal'

In [ ]:
# Create labels for the training and test sets
TrainLabel_Nor = np.zeros((TrainData_Nor.shape[0],2))
TrainLabel_Abn = np.zeros((TrainData_Abn.shape[0],2))
TestLabel_Nor  = np.zeros((TestData_Nor.shape[0],2))
TestLabel_Abn  = np.zeros((TestData_Abn.shape[0],2))

TrainLabel_Nor[:,0] = 1  # [1,0]: Normal
TrainLabel_Abn[:,1] = 1  # [0,1]: Abnormal
TestLabel_Nor[:,0]  = 1  # [1,0]: Normal
TestLabel_Abn[:,1]  = 1  # [0,1]: Abnormal

print(TrainLabel_Nor.shape, TestLabel_Nor.shape)
print(TrainLabel_Abn.shape, TestLabel_Abn.shape)

## Prepare the final Data and Label for ML modeling


In [ ]:
# Combine the normal and abnormal data/labels
TrainData  = np.concatenate([TrainData_Nor , TrainData_Abn ], axis=0)
TestData   = np.concatenate([TestData_Nor  , TestData_Abn  ], axis=0)
TrainLabel = np.concatenate([TrainLabel_Nor, TrainLabel_Abn], axis=0)
TestLabel  = np.concatenate([TestLabel_Nor , TestLabel_Abn ], axis=0)

print(TrainData.shape,  TestData.shape)
print(TrainLabel.shape, TestLabel.shape)

.

.

.

.

.

.

.



## Grid search for MLP (Multi-Layer Perceptron) hyperparameters

### Prepare lists of hyperparameters for grid search

In [ ]:
# Hyperparameters for grid search
param_ActFn = ['tanh', 'relu'] # activation function
param_Layer = [2, 3]           # number of hiddent layers
param_Lrate = [0.001, 0.01]    # learning rate

# Fixed hyperparameters
noOfNeuron = 16
Epoch      = 200

# Calculate the number of cases
NoOfCases = len(param_ActFn) * len(param_Layer) * len(param_Lrate)
NoOfCases

In [ ]:
# Create an empty dataframe to store the accuracy results
Accuracy_df = pd.DataFrame(np.zeros(shape=(NoOfCases , 4)),
                           columns=['Activation Function', 'Num of hidden layer', 'Learning rate', 'Accuracy'])
Accuracy_df

### Define a function to create MLP models by inputting the hyperparameters for grid search

In [ ]:
def MLP_model(input_data, noOfNeuron, temp_actfn, temp_layer, temp_lrate):
    keras.backend.clear_session()  # Clearing the Keras backend session (initiating variables)

    model = keras.Sequential()
    model.add(keras.layers.InputLayer(shape=(input_data.shape[1],)))  # Input Layer

    # Adding hidden layers according to the input value of `temp_layer`
    for i in range(temp_layer):
        model.add(keras.layers.Dense(units=noOfNeuron, activation=temp_actfn, name=f'Hidden{i+1}'))  # Hidden Layers

    model.add(keras.layers.Dense(units=2, activation='softmax', name='Output'))  # Output Layer

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=temp_lrate),
                  loss=keras.losses.CategoricalCrossentropy(),
                  metrics=['accuracy'])
    return model

In [ ]:
# Callback 1: CheckProcess
EpochForPrint = 20

class CheckProcess(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        keras.callbacks.Callback()
        if epoch%EpochForPrint == 0:
            print("{} Epochs Train Acc. : {:.2f}%  ".format(epoch, logs["accuracy"]*100))

In [ ]:
# Callback 2: Early Stopping
EalryStop = keras.callbacks.EarlyStopping(
    monitor="accuracy", patience=40, restore_best_weights=True)

### Train the ANN models with different combinations of hyperparameters and save them

In [ ]:
# Initialize a count value to store the performance of each model
cnt = 0

# Iterate through all possible combinations of activation functions, hidden layers, and learning rates
for temp_actfn in param_ActFn:          # Select each activation function in the list
    for temp_layer in param_Layer:      # Select each hidden layer configuration in the list
        for temp_lrate in param_Lrate:  # Select each learning rate value in the list

            print(f"\n[Case {cnt+1}] Activation: '{temp_actfn}', Num of layers: {temp_layer}, Learning rate: {temp_lrate}")

            # Create, train, and validate a temporary ANN model with the current combination of hyperparameters
            temp_model = MLP_model(TrainData, noOfNeuron, temp_actfn, temp_layer, temp_lrate)
            temp_model.fit(TrainData, TrainLabel, epochs=Epoch, verbose=0, callbacks=[CheckProcess(), EalryStop])
            Loss, Accuracy = temp_model.evaluate(TestData,  TestLabel, verbose=0)

            # Save the temporary model to a file with a corresponding name
            temp_model_name = f'MLP_{temp_actfn}_L{temp_layer}_LR{temp_lrate:.4f}.keras'
            temp_model.save('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/GridSearch_MLP/' + temp_model_name)

            # Store the performance (accuracy) of the temporary model in the dataframe
            Accuracy_df.iloc[cnt, :] = [temp_actfn, temp_layer, temp_lrate, Accuracy]
            cnt += 1

### Confirm the grid search results

In [ ]:
# Confirm the result of grid search
Accuracy_df

In [ ]:
# Sort the Accuracy_df by 'Accuracy' column in descending order
Accuracy_df_sorted = Accuracy_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)

# Output the best case
print("[Best case]\nActivation Function: " + Accuracy_df_sorted.iloc[0, 0] +
      "\nHidden Layers: {}\nLearning Rate: {:.4f}\n\nAccuracy: {:.2f}".format(Accuracy_df_sorted.iloc[0, 1],
                                                                       Accuracy_df_sorted.iloc[0, 2],
                                                                       Accuracy_df_sorted.iloc[0, 3]))

In [ ]:
# Calculate mean and standard deviation accuracy for each activation function
mean_accuracy_ActFn = Accuracy_df.groupby(['Activation Function'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_ActFn

In [ ]:
# Calculate mean and standard deviation of accuracy for each hidden layer
mean_accuracy_Layer = Accuracy_df.groupby(['Num of hidden layer'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_Layer

In [ ]:
# Calculate mean and standard deviation of accuracy for each learning rate
mean_accuracy_Lrate = Accuracy_df.groupby(['Learning rate'])['Accuracy'].agg(['mean', 'std']).reset_index()
mean_accuracy_Lrate

### Visualize the performance comparison for the selected hyperparameter

In [ ]:
# Set an index to select a hyperparmeter
# 0: activation function // 1: number of hidden layers // 2: learning rate
idx = 1

# Automatically define variables based on the selected index
H_Param = ['ActFn', 'Layer', 'Lrate']
H_Param_name = ['Activation Function', 'Num of hidden layer', 'Learning Rate']
Selected = H_Param[idx]
Selected_name = H_Param_name[idx]
exec('Result = mean_accuracy_' + H_Param[idx])

xLabel = Result.iloc[:, 0]
x_pos = np.arange(Result.shape[0])
y_val = Result['mean']
y_err = Result['std']

# Draw a bar chart to compare the model performance (diagnostic accuracy) for each hyperparameter
fig, ax = plt.subplots(figsize=(5, 4))

# Create a bar plot with error bars
ax.bar(x_pos, y_val, yerr=y_err, align='center', alpha=0.5, ecolor='black', capsize=10,
       color=['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple'])
ax.set_ylabel('Accuracy (mean)', fontsize=12)
ax.set_title(f"Performance comparsion by '{Selected_name}'\n", fontsize=14)
ax.set_xticks(x_pos)
ax.set_xticklabels(xLabel, fontsize=12)
ax.yaxis.grid()
ax.set_ylim([0.6, 1.0])

plt.tight_layout()
plt.show()

## Confusion matrix of the best MLP model

- TP: The number of instances where the model correctly predicted the positive class.
- TN: The number of instances where the model correctly predicted the negative class.
- FP: The number of instances where the model falsely predicted the positive class (actual negative instances).
- FN: The number of instances where the model falsely predicted the negative class (actual positive instances).

In [ ]:
# Retrieve activation function, hidden layers, and learning rate values from the first row of 'Accuracy_df_sorted'
Best_ActFn = Accuracy_df_sorted.iloc[0, 0]
Best_Layer = int(Accuracy_df_sorted.iloc[0, 1])
Best_Lrate = Accuracy_df_sorted.iloc[0, 2]

# Load the best MLP model using the retrieved hyperparameters
best_model_name = f'MLP_{Best_ActFn}_L{Best_Layer}_LR{Best_Lrate:.4f}.keras'
best_model = keras.models.load_model('/content/drive/MyDrive/Colab Notebooks/SavedFiles/ML_Models/GridSearch_MLP/' + best_model_name)

# Predict the output (Robotic spot-welding condition) for the test data
Predicted = best_model.predict(TestData)

# Convert TestLabel and Predicted into vectors to calculate the confusion matrix and evaluation metrics
TestLabel_rev = np.argmax(TestLabel, axis=1)
Predicted_rev = np.argmax(Predicted, axis=1)

# Plot the confusion matrix
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Calculate the confusion matrix
cm = confusion_matrix(TestLabel_rev, Predicted_rev)

plt.figure(figsize=(6, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap=plt.cm.Blues, cbar=False, square=True)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Confusion Matrix of the Best MLP Model")
plt.show()

## Evaluation metrics of the best MLP model

1. $Accuracy$: The proportion of correctly classified instances out of the total instances. It measures the overall performance of a classification model.

    - $Accuracy: (TP + TN) / (TP + TN + FP + FN)$

2. $Precision$: The proportion of true positive instances among the instances predicted as positive. It measures how well the model correctly identifies positive instances.

    - $Precision: TP / (TP + FP)$

3. $Recall$: The proportion of true positive instances among the actual positive instances. It measures the ability of the model to find all the positive instances.

    - $Recall: TP / (TP + FN)$

4. $F1 Score$: The harmonic mean of precision and recall. It provides a single score that balances both precision and recall, which is especially useful when dealing with imbalanced datasets.

    - $F1 Score: 2 * (Precision * Recall) / (Precision + Recall)$

In [ ]:
from sklearn import metrics

# Calculate the evaluation metrics
accuracy  = metrics.accuracy_score(TestLabel_rev, Predicted_rev)
precision = metrics.precision_score(TestLabel_rev, Predicted_rev)
recall    = metrics.recall_score(TestLabel_rev, Predicted_rev)
f1_score  = metrics.f1_score(TestLabel_rev, Predicted_rev)

# Print the evaluation metrics
print(f"Best ANN Model Evaluation:\n")
print(f"Accuracy : {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall   : {recall:.2f}")
print(f"F1 Score : {f1_score:.2f}")